# 02 Queries
Codecademy: Intro to SQL, lesson 2. Querying a single table.

Own dataset instead of Codecademy's `movies`: 31 European stocks in `data\stocks.db`
(built by `data\create_stocks.sql`; numbers are illustrative, not real market data).

In [43]:
# turns on the %%sql cell magic
%load_ext sql
# opens data\stocks.db (already filled with the stocks table)
%sql sqlite:///../data/stocks.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 1. Introduction

**Querying** = asking the database a question; the answer comes back as a result set.

Dataset, compared with Codecademy's:

| `stocks` | type | like `movies` column |
|---|---|---|
| `id` | INTEGER | `id` |
| `ticker`, `company` | TEXT | `name` |
| `sector`, `country` | TEXT | `genre` (1 sector is `NULL`) |
| `price`, `pe_ratio` | REAL | `imdb_rating` |
| `dividend_yield` | REAL, % | (`NULL` = no dividend) |
| `year_listed` | INTEGER | `year` |

Task: view the whole table and get to know the columns.

In [44]:
%%sql
SELECT * FROM stocks;

Running query in 'sqlite:///../data/stocks.db'

id,ticker,company,sector,country,price,pe_ratio,dividend_yield,year_listed
1,EBS,Erste Group,Financials,Austria,62.4,9.1,4.3,1997
2,RBI,Raiffeisen Bank International,Financials,Austria,24.8,4.5,None,2005
3,VIG,Vienna Insurance Group,Financials,Austria,30.5,8.7,4.6,1994
4,OMV,OMV,Energy,Austria,45.1,7.8,6.5,1987
5,VER,Verbund,Utilities,Austria,68.9,15.2,3.9,1988
6,ANDR,Andritz,Industrials,Austria,58.2,13.4,4.1,2001
7,VOE,voestalpine,Materials,Austria,24.3,11.9,2.9,1995
8,WIE,Wienerberger,Materials,Austria,31.6,16.8,2.8,1869
9,POST,Oesterreichische Post,Industrials,Austria,30.2,15.0,5.8,2006
10,TKA,Telekom Austria,Telecom,Austria,8.1,9.8,4.8,2000


## 2. Select

Several columns: list them separated by commas. Line breaks don't matter.

pandas: `df[['ticker', 'company', 'sector']]`

In [45]:
%%sql
SELECT ticker, company, sector
FROM stocks;

Running query in 'sqlite:///../data/stocks.db'

ticker,company,sector
EBS,Erste Group,Financials
RBI,Raiffeisen Bank International,Financials
VIG,Vienna Insurance Group,Financials
OMV,OMV,Energy
VER,Verbund,Utilities
ANDR,Andritz,Industrials
VOE,voestalpine,Materials
WIE,Wienerberger,Materials
POST,Oesterreichische Post,Industrials
TKA,Telekom Austria,Telecom


## 3. As

`AS` gives a column an **alias**: a new name in the result only. The table itself is not changed.
Useful when a column name is unclear, or for a cleaner report.

pandas: `df.rename(columns={'pe_ratio': 'P/E'})`

Quotes: an alias is a name, so standard SQL uses double quotes or none (`AS "P/E"`, `AS pe`).
Codecademy (SQLite) recommends single quotes; they work in SQLite but not in PostgreSQL.

In [46]:
%%sql
SELECT ticker AS 'Ticker',
       pe_ratio AS 'P/E',
       dividend_yield AS 'Dividend %'
FROM stocks;

Running query in 'sqlite:///../data/stocks.db'

Ticker,P/E,Dividend %
EBS,9.1,4.3
RBI,4.5,None
VIG,8.7,4.6
OMV,7.8,6.5
VER,15.2,3.9
ANDR,13.4,4.1
VOE,11.9,2.9
WIE,16.8,2.8
POST,15.0,5.8
TKA,9.8,4.8


## 4. Distinct

`DISTINCT` returns each value only once (filters out duplicates). Quick way to see which categories exist.

- `NULL` counts as a value: Porr's missing sector shows up as one empty row
- With several columns, `DISTINCT` keeps each unique **combination**: `SELECT DISTINCT sector, country`

pandas: `df['sector'].unique()`

Task: list the unique sectors, then the unique countries.

In [47]:
%%sql
SELECT DISTINCT sector
FROM stocks;   -- 9 sectors + 1 empty (NULL)

Running query in 'sqlite:///../data/stocks.db'

sector
Financials
Energy
Utilities
Industrials
Materials
Telecom
None
Technology
Consumer
Healthcare


In [48]:
%%sql
SELECT DISTINCT country
FROM stocks;

Running query in 'sqlite:///../data/stocks.db'

country
Austria
Germany
Netherlands
France
Switzerland
Denmark
Sweden
Czechia


## 5. Where

`WHERE` keeps only the rows where a condition is true. An **operator** builds the condition:

| Operator | Means |
|---|---|
| `=` / `!=` | equal / not equal |
| `>` / `<` | greater / less than |
| `>=` / `<=` | greater / less than or equal |

pandas: `df[df['pe_ratio'] < 10]`

Task: find cheaply valued stocks (P/E below 10), then recent listings (after 2010).

In [49]:
%%sql
SELECT *
FROM stocks
WHERE pe_ratio < 10;

Running query in 'sqlite:///../data/stocks.db'

id,ticker,company,sector,country,price,pe_ratio,dividend_yield,year_listed
1,EBS,Erste Group,Financials,Austria,62.4,9.1,4.3,1997
2,RBI,Raiffeisen Bank International,Financials,Austria,24.8,4.5,None,2005
3,VIG,Vienna Insurance Group,Financials,Austria,30.5,8.7,4.6,1994
4,OMV,OMV,Energy,Austria,45.1,7.8,6.5,1987
10,TKA,Telekom Austria,Telecom,Austria,8.1,9.8,4.8,2000
11,PORR,Porr,None,Austria,20.0,9.0,3.5,1986
17,VOW3,Volkswagen,Consumer,Germany,90.0,4.0,6.5,1961
22,TTE,TotalEnergies,Energy,France,56.0,8.0,5.5,1991
23,BNP,BNP Paribas,Financials,France,62.0,7.0,7.4,1993


In [50]:
%%sql
SELECT *
FROM stocks
WHERE year_listed > 2010;

Running query in 'sqlite:///../data/stocks.db'

id,ticker,company,sector,country,price,pe_ratio,dividend_yield,year_listed
18,ZAL,Zalando,Consumer,Germany,28.0,25.0,None,2014
20,ADYEN,Adyen,Technology,Netherlands,1450.0,50.0,None,2018
30,SPOT,Spotify Technology,Technology,Sweden,500.0,80.0,None,2018


## 6. Like

`LIKE` searches text for a **pattern**, using wildcards:

| Wildcard | Matches | Example | Finds |
|---|---|---|---|
| `_` | exactly 1 character | `'NOV_'` | NOVN, not NOVO-B |
| `%` | 0 or more characters | `'S%'` | starts with S |
| | | `'%Group%'` | contains "Group" |

- `LIKE` is case-insensitive in SQLite: `'%group%'` also finds "Erste Group". `=` is case-sensitive (project: the `'ELena'` typo)
- Word at the start: include the space, `'The %'`, otherwise "Theory" would match too

pandas: `df[df['company'].str.contains('Group')]`

In [51]:
%%sql
SELECT * FROM stocks
WHERE ticker LIKE 'NOV_';      -- Novartis only

Running query in 'sqlite:///../data/stocks.db'

id,ticker,company,sector,country,price,pe_ratio,dividend_yield,year_listed
26,NOVN,Novartis,Healthcare,Switzerland,95.0,15.0,3.7,1996


In [52]:
%%sql
SELECT * FROM stocks
WHERE company LIKE '%group%';  -- Erste, Vienna Insurance, UBS, CEZ

Running query in 'sqlite:///../data/stocks.db'

id,ticker,company,sector,country,price,pe_ratio,dividend_yield,year_listed
1,EBS,Erste Group,Financials,Austria,62.4,9.1,4.3,1997
3,VIG,Vienna Insurance Group,Financials,Austria,30.5,8.7,4.6,1994
28,UBSG,UBS Group,Financials,Switzerland,27.0,18.0,2.6,1998
31,CEZ,CEZ Group,Utilities,Czechia,40.0,12.0,6.0,1993


## 7. Is Null

Missing values (`NULL`) can't be tested with `=` or `!=`. Use `IS NULL` / `IS NOT NULL`.

pandas: `df[df['dividend_yield'].isna()]`

Task: find the stocks that pay no dividend.

In [53]:
%%sql
SELECT ticker, company
FROM stocks
WHERE dividend_yield IS NULL;   -- RBI, Zalando, Adyen, Spotify

Running query in 'sqlite:///../data/stocks.db'

ticker,company
RBI,Raiffeisen Bank International
ZAL,Zalando
ADYEN,Adyen
SPOT,Spotify Technology


## 8. Between

`BETWEEN x AND y` keeps values in a range, **both limits included**. Works for numbers, text and dates.

pandas: `df[df['year_listed'].between(1990, 1999)]`

In [54]:
%%sql
SELECT ticker, company, year_listed
FROM stocks
WHERE year_listed BETWEEN 1990 AND 1999;

Running query in 'sqlite:///../data/stocks.db'

ticker,company,year_listed
EBS,Erste Group,1997
VIG,Vienna Insurance Group,1994
VOE,voestalpine,1995
DTE,Deutsche Telekom,1996
ASML,ASML Holding,1995
TTE,TotalEnergies,1991
BNP,BNP Paribas,1993
NOVN,Novartis,1996
UBSG,UBS Group,1998
CEZ,CEZ Group,1993


**Mistake 1: text upper limit.** Text sorts letter by letter, so `'F'` < `'Fa...'`. The limit `'F'` means "up to exactly F":

    WHERE name BETWEEN 'D' AND 'F'    -- D, E only (Frozen is after 'F', excluded)
    WHERE name BETWEEN 'D' AND 'G'    -- D, E, F   (correct)

Same rule as numbers: `BETWEEN 1970 AND 1979` excludes 1979.5, and `'Frozen'` is past `'F'` the same way.

**Mistake 2: `BETWEEN` on text is case-sensitive** (unlike `LIKE`). Uppercase A-Z sorts before lowercase a-z,
so `BETWEEN 'a' AND 'z'` finds only lowercase names. In our data, "voestalpine" is missing below:

**Mistake 3: quoted numbers.** I wrote `BETWEEN '1970' AND '1979'`. Quotes make it text; SQLite converts it silently,
other databases may not. Numbers go without quotes.

In [55]:
%%sql
SELECT company
FROM stocks
WHERE company BETWEEN 'U' AND 'W';   -- UBS, Verbund, Vienna Insurance, Volkswagen; NOT voestalpine (lowercase v)

Running query in 'sqlite:///../data/stocks.db'

company
Vienna Insurance Group
Verbund
Volkswagen
UBS Group


## 9. And

`AND` combines conditions: **all** must be true for a row to be kept.

pandas: `df[(df['sector'] == 'Financials') & (df['dividend_yield'] > 4)]`

Task: high-dividend financials, then cheap Austrian stocks.

In [56]:
%%sql
SELECT ticker, company, dividend_yield
FROM stocks
WHERE sector = 'Financials'
  AND dividend_yield > 4;

Running query in 'sqlite:///../data/stocks.db'

ticker,company,dividend_yield
EBS,Erste Group,4.3
VIG,Vienna Insurance Group,4.6
ALV,Allianz,5.0
BNP,BNP Paribas,7.4


In [57]:
%%sql
SELECT ticker, company, pe_ratio
FROM stocks
WHERE country = 'Austria'
  AND pe_ratio < 10;

Running query in 'sqlite:///../data/stocks.db'

ticker,company,pe_ratio
EBS,Erste Group,9.1
RBI,Raiffeisen Bank International,4.5
VIG,Vienna Insurance Group,8.7
OMV,OMV,7.8
TKA,Telekom Austria,9.8
PORR,Porr,9.0


## 10. Or

| Operator | Logic | Keeps a row if |
|---|---|---|
| `AND` | A ∧ B (A ∩ B) | all conditions true |
| `OR` | A ∨ B (A ∪ B) | at least one condition true |

pandas: `df[(df['sector'] == 'Technology') | (df['pe_ratio'] > 30)]`

In [58]:
%%sql
SELECT ticker, company, sector, pe_ratio
FROM stocks
WHERE sector = 'Technology'
   OR pe_ratio > 30;

Running query in 'sqlite:///../data/stocks.db'

ticker,company,sector,pe_ratio
SAP,SAP,Technology,45.0
ASML,ASML Holding,Technology,32.0
ADYEN,Adyen,Technology,50.0
SPOT,Spotify Technology,Technology,80.0


**Pitfall: `AND` is evaluated before `OR`.** `A OR B AND C` = A ∨ (B ∧ C). Use parentheses when you mean (A ∨ B) ∧ C.

Task: Austrian or German stocks with a dividend above 5%. The two cells below differ only in the parentheses:

In [59]:
%%sql
-- Wrong: = Austria ∨ (Germany ∧ dividend > 5) -> every Austrian stock, regardless of dividend
SELECT ticker, country, dividend_yield
FROM stocks
WHERE country = 'Austria' OR country = 'Germany'
  AND dividend_yield > 5;

Running query in 'sqlite:///../data/stocks.db'

ticker,country,dividend_yield
EBS,Austria,4.3
RBI,Austria,None
VIG,Austria,4.6
OMV,Austria,6.5
VER,Austria,3.9
ANDR,Austria,4.1
VOE,Austria,2.9
WIE,Austria,2.8
POST,Austria,5.8
TKA,Austria,4.8


In [60]:
%%sql
-- Right: (Austria ∨ Germany) ∧ dividend > 5
SELECT ticker, country, dividend_yield
FROM stocks
WHERE (country = 'Austria' OR country = 'Germany')
  AND dividend_yield > 5;

Running query in 'sqlite:///../data/stocks.db'

ticker,country,dividend_yield
OMV,Austria,6.5
POST,Austria,5.8
BAS,Germany,7.5
VOW3,Germany,6.5


## 11. Order By

`ORDER BY` sorts the result set. `ASC` = low to high / A-Z (default), `DESC` = high to low / Z-A.

    SELECT ... FROM ... WHERE ... ORDER BY ... LIMIT ...    <- clause order is fixed

- You can sort by a column that isn't selected
- SQLite puts `NULL` **first** in `ASC` order (and last in `DESC`)

pandas: `df.sort_values('dividend_yield', ascending=False)`

In [61]:
%%sql
SELECT ticker, company, dividend_yield
FROM stocks
ORDER BY dividend_yield DESC;   -- the 4 NULLs end up at the bottom

Running query in 'sqlite:///../data/stocks.db'

ticker,company,dividend_yield
BAS,BASF,7.5
BNP,BNP Paribas,7.4
OMV,OMV,6.5
VOW3,Volkswagen,6.5
CEZ,CEZ Group,6.0
POST,Oesterreichische Post,5.8
TTE,TotalEnergies,5.5
ALV,Allianz,5.0
TKA,Telekom Austria,4.8
VIG,Vienna Insurance Group,4.6


## 12. Limit

`LIMIT n` caps the result at `n` rows. Always the very last clause.
Combined with `ORDER BY` it gives a "top n" list.

Not supported everywhere: SQL Server uses `SELECT TOP 5 ...` instead.

pandas: `df.nlargest(5, 'dividend_yield')`

Task: the 5 highest dividend yields.

In [62]:
%%sql
SELECT ticker, company, dividend_yield
FROM stocks
ORDER BY dividend_yield DESC
LIMIT 5;

Running query in 'sqlite:///../data/stocks.db'

ticker,company,dividend_yield
BAS,BASF,7.5
BNP,BNP Paribas,7.4
OMV,OMV,6.5
VOW3,Volkswagen,6.5
CEZ,CEZ Group,6.0


## 13. Case

`CASE` is SQL's if-then-else. It creates a new column in the result.

    CASE
      WHEN condition_1 THEN 'value_1'   <- checked first
      WHEN condition_2 THEN 'value_2'
      ELSE 'value_3'                    <- if nothing above is true
    END AS 'Column name'                <- END is required; AS shortens the long default name

- The **first** true `WHEN` wins, so order matters: `> 5` must come before `> 2`
- `NULL` makes every comparison fail, so it falls through to `ELSE`. Handle it with its own `WHEN ... IS NULL`

pandas: `pd.cut(df['dividend_yield'], bins=[0, 2, 5, 100], labels=['Low', 'Medium', 'High'])`

Task: group stocks by dividend yield.

In [63]:
%%sql
SELECT ticker, company, dividend_yield,
  CASE
    WHEN dividend_yield IS NULL THEN 'No dividend'
    WHEN dividend_yield > 5 THEN 'High'
    WHEN dividend_yield > 2 THEN 'Medium'
    ELSE 'Low'
  END AS 'Dividend group'
FROM stocks;

Running query in 'sqlite:///../data/stocks.db'

ticker,company,dividend_yield,Dividend group
EBS,Erste Group,4.3,Medium
RBI,Raiffeisen Bank International,None,No dividend
VIG,Vienna Insurance Group,4.6,Medium
OMV,OMV,6.5,High
VER,Verbund,3.9,Medium
ANDR,Andritz,4.1,Medium
VOE,voestalpine,2.9,Medium
WIE,Wienerberger,2.8,Medium
POST,Oesterreichische Post,5.8,High
TKA,Telekom Austria,4.8,Medium


## 14. Review

All lesson 2 clauses in one query, in their fixed order: the 5 highest-yielding German or Austrian stocks with a P/E between 5 and 20.
Full list of key words: `00_key_words.md`.

In [64]:
%%sql
SELECT ticker,
       company AS 'Company',
       dividend_yield AS 'Dividend %',
       CASE
         WHEN pe_ratio < 10 THEN 'Cheap'
         ELSE 'Fair'
       END AS 'Valuation'
FROM stocks
WHERE (country = 'Austria' OR country = 'Germany')
  AND pe_ratio BETWEEN 5 AND 20
ORDER BY dividend_yield DESC
LIMIT 5;

Running query in 'sqlite:///../data/stocks.db'

ticker,Company,Dividend %,Valuation
BAS,BASF,7.5,Fair
OMV,OMV,6.5,Cheap
POST,Oesterreichische Post,5.8,Fair
ALV,Allianz,5.0,Fair
TKA,Telekom Austria,4.8,Cheap
